In [1]:
pip install azure-ai-documentintelligence azure-ai-openai psycopg2-binary pdfminer.six pyvis streamlit


   ---------------------------------------- 0.0/5.6 MB ? eta -:--:--
   ---------------------------------------  5.5/5.6 MB 196.8 MB/s eta 0:00:01
   ---------------------------------------  5.5/5.6 MB 196.8 MB/s eta 0:00:01
   ---------------------------------------  5.5/5.6 MB 196.8 MB/s eta 0:00:01
   ---------------------------------------- 5.6/5.6 MB 8.0 MB/s  0:00:00
   ---------------------------------------- 0.0/756.0 kB ? eta -:--:--
   -------------------------- ----------- 524.3/756.0 kB 195.1 MB/s eta 0:00:01
   -------------------------- ----------- 524.3/756.0 kB 195.1 MB/s eta 0:00:01
   ---------------------------------------- 756.0/756.0 kB 921.9 kB/s  0:00:00
   ---------------------------------------- 0.0/9.0 MB ? eta -:--:--
   -------- ------------------------------- 1.8/9.0 MB 202.9 MB/s eta 0:00:01
   ---------------------------------------  8.9/9.0 MB 22.7 MB/s eta 0:00:01
   ---------------------------------------  8.9/9.0 MB 22.7 MB/s eta 0:00:01
   ----------


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import psycopg2
import openai
from openai import AzureOpenAI
import numpy as np
import os
import json


In [ ]:
AZURE_OPENAI_API_KEY = ""
AZURE_OPENAI_ENDPOINT = "https://tin-accel-openai-rrqqzdcm76yg2.openai.azure.com/"
EMBEDDING_MODEL = "text-embedding-3-large"
POSTGRES_CONN = {
    "dbname": "mygraphdb",
    "user": "postgres",
    "password": "postgres",
    "host": "localhost",
    "port": 5432
}

client = AzureOpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_version="2023-07-01-preview"
)

In [30]:
print(client.base_url)

https://tin-accel-openai-rrqqzdcm76yg2.openai.azure.com/openai/


In [4]:
# --------------------------
# Connect to Postgres / AGE
# --------------------------
conn = psycopg2.connect(**POSTGRES_CONN)
cur = conn.cursor()

# Load AGE extension and set search path
cur.execute("LOAD 'age';")
cur.execute("SET search_path = ag_catalog, \"$user\", public;")
conn.commit()



In [45]:
cur.execute("SELECT create_graph('doc_graph');")
conn.commit()

In [5]:
# --------------------------
# Function: Generate Embeddings
# --------------------------
# openai.api_type = "azure"
# openai.api_version = "2023-07-01-preview"
# openai.api_key = AZURE_OPENAI_API_KEY
# openai.api_base = AZURE_OPENAI_ENDPOINT

def get_embedding(text: str):
    response = client.embeddings.create(
        model=EMBEDDING_MODEL,  # e.g., 'text-embedding-3-large'
        input=text
    )
    return response.data[0].embedding



In [32]:
embedding = get_embedding("Sample text for embedding")
print(len(embedding), embedding[:10])

3072 [-0.0017872226890176535, -0.025673523545265198, -0.01591501198709011, -0.0028554212767630816, 0.028264766559004784, -0.007746163289994001, -0.029973885044455528, 0.04157015681266785, -0.012698929756879807, -0.008518022485077381]


In [6]:
# --------------------------
# Function: Add Document Chunks to Graph
# --------------------------
def add_chunk_to_graph(graph_name, chunk_id, text, embedding):
    # Convert embedding to string for SQL insertion
    embedding_str = '[' + ','.join(map(str, embedding)) + ']'

    query = f"""
    SELECT * FROM cypher('{graph_name}', $$
        CREATE (:DocumentChunk {{
            text: '{text.replace("'", "''")}',
            id: {chunk_id},
            embedding: '{embedding_str}'
        }})
    $$) AS t(node agtype);
    """
    cur.execute(query)
    conn.commit()



In [42]:
# --------------------------
# Sample Document Split
# --------------------------
document = """
Bob runs a burger joint in town. Linda helps him at the restaurant.
Tina, Gene, and Louise all have their own personalities.
The family often gets into funny adventures together.
"""

chunks = [s.strip() for s in document.split('\n') if s.strip()]

print(chunks)



['Bob runs a burger joint in town. Linda helps him at the restaurant.', 'Tina, Gene, and Louise all have their own personalities.', 'The family often gets into funny adventures together.']


In [46]:
# --------------------------
# Insert Chunks as Nodes
# --------------------------
for idx, chunk in enumerate(chunks, start=1):
    try:
        embedding = get_embedding(chunk)
        add_chunk_to_graph("doc_graph", idx, chunk, embedding)
        conn.commit()  # ✅ commit each successful insert
    except Exception as e:
        print(f"Error on chunk {idx}: {e}")
        conn.rollback()  # ✅ rollback so next iteration can continue

print("All chunks inserted into AGE graph!")

# Close connection
cur.close()
conn.close()

All chunks inserted into AGE graph!


### Pausing here

Currently done
* read text
* embed text
* build doc graph in postgres

left to be done,
* build relationships using llm and building detailed graph

In [8]:
cur.execute("""
SELECT n::jsonb
FROM cypher('doc_graph', $$
    MATCH (n:DocumentChunk)
    RETURN n
$$) AS (n agtype);
""")

rows = cur.fetchall()

chunks = []
for row in rows:
    node = row[0]  # Already parsed JSON if ::jsonb works
    if isinstance(node, dict):  
        chunks.append(node["properties"]["text"])
    else:
        # Fallback if ::jsonb not supported and raw agtype text returned
        node_str = str(node).replace("::vertex", "").strip()
        try:
            node_json = json.loads(node_str)
            chunks.append(node_json["properties"]["text"])
        except json.JSONDecodeError:
            print("⚠️ Could not parse:", node_str[:200])
            continue

print(f"✅ Retrieved {len(chunks)} text chunks from doc_graph")



CannotCoerce: cannot cast type agtype to jsonb
LINE 2: SELECT n::jsonb
                ^


In [ ]:
from openai import AzureOpenAI

client = AzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version="2024-05-01-preview"
)

prompt = f"""
You are an expert knowledge graph builder.
Given these text snippets, identify explicit or implicit relationships between them.

Each relationship should be output as JSON objects with:
- source_text (the text of the source node)
- relation (the relationship label)
- target_text (the text of the target node)

Text snippets:
{json.dumps(chunks, indent=2)}

Example Output:
[
  {{
    "source_text": "Bob owns Bob's Burgers restaurant.",
    "relation": "OWNS",
    "target_text": "Bob's Burgers restaurant is located in Seymour's Bay."
  }}
]
"""

response = client.chat.completions.create(
    model="gpt-4o",
    messages=[{"role": "user", "content": prompt}],
    temperature=0.2
)

relationships_json = json.loads(response.choices[0].message.content)


In [ ]:
for rel in relationships_json:
    source_text = rel["source_text"].replace("'", "''")
    target_text = rel["target_text"].replace("'", "''")
    relation = rel["relation"].upper().replace(" ", "_")

    query = f"""
    SELECT * FROM cypher('doc_graph', $$
        MATCH (a:DocumentChunk), (b:DocumentChunk)
        WHERE a.text = '{source_text}' AND b.text = '{target_text}'
        CREATE (a)-[:{relation}]->(b)
    $$) AS (edge agtype);
    """
    try:
        cur.execute(query)
        conn.commit()
    except Exception as e:
        print(f"Failed to insert edge: {e}")
        conn.rollback()
